# 10-K RAG 베이스라인 평가

## 이 노트북을 만든 이유

`storereg.ipynb`에서 naive RAG를 만들고 질문을 몇 개 던져봤더니 답이 잘 나왔습니다.

그런데 과제는 이렇게 하라고 합니다.

> 베이스라인을 질문으로 직접 테스트하면서 **어디가 부족한지 관찰**하세요.
> 부족한 지점이 보이면, 그 **증상에 맞는 처방만** 골라 적용하세요.

여기서 막힙니다. **"잘 나오는데 뭐가 부족한지 어떻게 알죠?"**

이 노트북은 그 질문에 답하는 과정입니다.
결론부터 코드로 주지 않고, 한 단계씩 직접 돌려보면서
"아 이래서 이게 필요하구나"를 느끼는 순서로 갑니다.

---

## 진행 방식

각 단계는 이렇게 구성됩니다.

```
[직접 돌려본다] → [결과를 본다] → [불편한 점을 발견한다] → [그래서 다음 코드를 쓴다]
```

**납득이 안 되는 단계가 나오면 멈추고 질문해주세요.**
설명을 읽고 "왜?"가 남으면 그 상태로 다음으로 넘어가지 마세요.

---

# 0단계: 준비 — 베이스라인을 다시 만든다

평가를 하려면 **평가 대상이 고정**되어 있어야 합니다.
`storereg.ipynb`와 똑같은 파이프라인을 이 노트북 안에 다시 만듭니다.

(다른 노트북의 변수를 가져다 쓰면, 그 노트북을 수정했을 때 평가 결과가 조용히 바뀝니다.)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PDF_PATH = ROOT / "data" / "row" / "KLA-10-K-2026.pdf"

print("PDF 경로 :", PDF_PATH)
print("존재 여부 :", PDF_PATH.exists())

### PDF를 읽고 자르고 임베딩합니다

`storereg.ipynb`와 **완전히 동일한 설정**입니다.
- 파서: `PyPDFLoader` (단순 텍스트 추출, 표 구조 인식 없음)
- 청킹: 고정 1000자, 200자 겹침
- 임베딩: `text-embedding-3-small`

이 셀은 임베딩 API를 호출하므로 **1~2분** 걸립니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

docs = PyPDFLoader(str(PDF_PATH)).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
chunks = splitter.split_documents(docs)

print(f"페이지 수 : {len(docs)}")
print(f"청크 수   : {len(chunks)}")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = InMemoryVectorStore(embedding=embeddings)
vectorstore.add_documents(chunks)

print("벡터스토어 준비 완료")

---

# 1단계: 지금 방식대로 질문해본다

먼저 **지금 하고 계신 방식** 그대로 해봅니다.
질문을 던지고, 나온 답을 눈으로 읽습니다.

In [ ]:
# 1단계: 지금까지 하던 방식 — 질문하고 답을 눈으로 본다

question = "What was KLA's total revenue in fiscal year 2026?"

docs_found = vectorstore.similarity_search(question, k=5)

for i, d in enumerate(docs_found, 1):
    print(f"--- {i}등 ---")
    print(d.page_content[:200].replace("\n", " "))
    print()

### 여기서 첫 번째 불편함

위 출력을 보고 **"이게 맞게 나온 건가?"** 를 판단하셨을 겁니다.

판단하려면 이렇게 해야 합니다.

1. 5개 청크를 다 읽는다
2. 정답(총매출)이 들어있는 청크가 있는지 찾는다
3. 있으면 몇 번째인지 센다

질문 1개에 이 과정이 필요합니다.
**질문 15개면 75개 청크를 읽어야 합니다.**

그리고 나중에 BM25를 추가한 뒤 "좋아졌나?"를 보려면
**또 75개를 읽고 아까 것과 비교**해야 합니다.

> 이건 사람이 할 수 있는 일이 아닙니다.

---

# 2단계: 자동으로 판정하게 만든다

사람이 눈으로 세는 걸 코드가 대신하게 만들어야 합니다.

문제는 **"정답이 들어있다"를 코드가 어떻게 아느냐** 입니다.

가장 단순한 방법: **정답 청크에 반드시 있어야 할 문자열을 미리 정해둔다.**

FY2026 총매출을 물었다면, 정답 근거 청크에는 반드시 `13,579,476`이 있어야 합니다.
- 검색 결과에 이 문자열이 있으면 → 검색 성공
- 없으면 → 실패

먼저 이 숫자가 진짜 문서에 있는지부터 확인합니다.
(없는 숫자로 채점하면 전부 실패로 나와서 진단이 무의미해집니다.)

In [ ]:
# 2단계: 정답 문자열이 실제 문서에 있는지 먼저 확인한다

all_text = "\n".join(c.page_content for c in chunks)

for kw in ["13,579,476", "4,830,771", "Milpitas", "KLAC"]:
    print(f"{kw:15s} 문서에 존재: {kw in all_text}")

### 판정 함수를 만들어봅니다

이제 "검색해서 → 정답 문자열이 몇 번째에 있는지" 세는 함수를 씁니다.

In [ ]:
# 2단계: 사람 대신 코드가 세게 만든다

def check(query: str, answer_keyword: str, k: int = 5):
    docs_found = vectorstore.similarity_search(query, k=k)
    for rank, d in enumerate(docs_found, start=1):
        if answer_keyword in d.page_content:
            return f"{rank}등에서 발견"
    return "못 찾음"


print(check("What was KLA's total revenue in fiscal year 2026?", "13,579,476"))
print(check("Where is KLA headquartered?", "Milpitas"))

### 두 번째 불편함

판정은 자동화됐습니다. 그런데 이걸로 뭘 알 수 있나요?

`"1등에서 발견"` — 좋은 건 알겠는데,
**질문 15개를 돌리면 문자열 15줄이 나옵니다. 그래서 전체적으로 좋은 건가요?**

"1등, 3등, 못 찾음, 2등, 1등, ..." 을 보고
**"베이스라인이 몇 점짜리다"** 라고 말할 수 없습니다.

리포트에 "처방 전 → 후" 비교표를 쓰려면 **숫자 하나**가 필요합니다.

---

# 3단계: 점수로 바꾼다

문자열 대신 숫자로 만듭니다. 두 가지를 씁니다.

### Hit Rate — "찾았나?"

상위 5개 안에 정답이 있으면 1, 없으면 0.
전체 평균이 `0.6`이면 → **질문 10개 중 4개는 근거 없이 답한 것**입니다.

### MRR — "몇 등으로 찾았나?"

1등이면 `1.0`, 2등이면 `0.5`, 3등이면 `0.33`, 못 찾으면 `0`.
(등수의 역수라서 Reciprocal Rank입니다.)

### 왜 둘 다 필요한가

이게 핵심입니다. **두 숫자를 같이 봐야 처방이 갈립니다.**

| Hit Rate | MRR | 해석 | 처방 |
|---|---|---|---|
| 낮음 | 낮음 | 애초에 **못 찾음** | 파서 교체 / BM25 추가 |
| 높음 | 낮음 | 찾긴 하는데 **순위가 밀림** | **재랭킹** |

과제 메뉴의 *"대충 관련은 있는데 진짜 필요한 건 3~4번째 순위에 있다"* 가
정확히 두 번째 줄입니다. **MRR 없이는 이 증상을 발견할 수 없습니다.**

In [ ]:
# 3단계: 등수를 점수로 바꾼다

def score(query: str, answer_keywords: list[str], k: int = 5):
    docs_found = vectorstore.similarity_search(query, k=k)
    for rank, d in enumerate(docs_found, start=1):
        if any(kw in d.page_content for kw in answer_keywords):
            return dict(hit=1, rank=rank, rr=1.0 / rank)
    return dict(hit=0, rank=None, rr=0.0)


print(score("What was KLA's total revenue in fiscal year 2026?", ["13,579,476"]))
print(score("Where is KLA headquartered?", ["Milpitas"]))

### 세 번째 불편함 — 가장 중요한 지점

이제 점수가 나옵니다. 질문 15개를 넣으면 평균도 낼 수 있습니다.

**그런데 어떤 질문 15개를 넣을 건가요?**

여기서 대부분 실수합니다. 떠오르는 대로 질문을 씁니다.

- "KLA는 뭐 하는 회사야?"
- "KLA의 주요 사업은?"
- "KLA가 만드는 제품은?"

이렇게 15개를 만들면 **거의 100점이 나옵니다.**
서술형 개요 질문은 naive RAG가 **가장 잘하는 영역**이기 때문입니다.

> 그러면 "우리 시스템 완벽함"이라는 **틀린 결론**이 나옵니다.
> 지금 "값이 엄청 잘 나온다"고 느끼시는 것도 이 때문일 가능성이 큽니다.

직접 확인해봅시다.

In [ ]:
# 3단계: 쉬운 질문만 넣으면 어떻게 되는지 직접 본다

easy_questions = [
    ("What does KLA do as a business?", ["process control", "inspection"]),
    ("What are KLA's main products?", ["inspection", "metrology"]),
    ("What markets does KLA serve?", ["semiconductor"]),
    ("Where is KLA headquartered?", ["Milpitas"]),
]

for q, kws in easy_questions:
    r = score(q, kws)
    print(f"{'HIT ' + str(r['rank']) + '등' if r['hit'] else 'MISS':12s} | {q}")

### 전부 통과했을 겁니다

이게 함정입니다. 이 결과만 보면 처방이 하나도 필요 없어 보입니다.

이번엔 **표 안의 숫자**를 물어봅니다.
`PyPDFLoader`는 표를 그냥 텍스트로 뭉개서 읽기 때문에,
여기가 깨질 가능성이 높은 지점입니다.

In [ ]:
# 3단계: 이번엔 깨질 만한 곳을 찌른다 — 표 안의 숫자

hard_questions = [
    ("What was KLA's total revenue in fiscal year 2026?", ["13,579,476"]),
    ("What was KLA's net income in fiscal year 2026?", ["4,830,771"]),
    ("What was KLA's R&D expense in fiscal 2026?", ["1,532,118"]),
    ("What is KLA's IRS Employer Identification Number?", ["04-2564110"]),
]

for q, kws in hard_questions:
    r = score(q, kws)
    print(f"{'HIT ' + str(r['rank']) + '등' if r['hit'] else 'MISS':12s} | {q}")

### 두 결과를 비교해보세요

쉬운 질문과 어려운 질문의 결과가 다르다면,
**질문을 어떻게 고르느냐에 따라 결론이 완전히 달라진다**는 뜻입니다.

> 그래서 테스트 질문은 **아무거나 15개**가 아니라,
> **깨질 만한 곳을 의도적으로 찌르도록 설계**해야 합니다.

---

# 4단계: 질문 세트를 설계한다

이제 질문을 체계적으로 만듭니다.

## 어떻게 설계하는가

과제의 "증상별 처방 메뉴"를 **거꾸로** 읽습니다.
각 처방이 고치려는 증상이 있고, 그 증상을 유발하는 질문 유형이 있습니다.

| 카테고리 | 무엇을 찌르는가 | 실패하면 의심할 처방 |
|---|---|---|
| `narrative` | 서술형 개념 | **대조군** (이건 통과해야 정상) |
| `table_numeric` | 재무제표 **표 안의 숫자** | 레이아웃 파서 / 표 별도 청킹 |
| `exact_term` | 정확한 **고유명사·식별자** | BM25 / 하이브리드 검색 |
| `cross_section` | 여러 섹션을 **걸쳐야** 답 가능 | 섹션 청킹 / Parent-Child |
| `multi_year` | 여러 연도 숫자 **비교** | 표 처방 + 재랭킹 |

### `narrative`를 넣는 이유 — 대조군

이건 **통과하라고** 넣습니다.

- `narrative`까지 실패 → 파이프라인 자체가 고장난 것 (환경 문제부터 확인)
- `narrative`만 통과 → "쉬운 질문만 되는" 상태 (처방 필요)

비교 기준이 없으면 "원래 이 정도인가?"를 판단할 수 없습니다.

### 자료 구조를 정하고 갑니다

질문마다 4개 정보가 필요합니다: `id`, `category`, `q`, `expect_keywords`.

`dict(...)`로 써도 동작하지만, 타입 검사기(Pylance)가
값 타입을 `str | list[str]`로 뭉뚱그려 추론해서 경고를 냅니다.

`TypedDict`로 키마다 타입을 명시하면 경고가 사라집니다.
**런타임 동작은 완전히 동일합니다.**

In [ ]:
# 4단계: 질문 세트의 자료 구조 정의

from typing import TypedDict


class Question(TypedDict):
    id: str
    category: str
    q: str
    expect_keywords: list[str]

### 질문 15개를 카테고리별로 작성합니다

`expect_keywords`의 숫자들은 **제가 지어낸 게 아니라**
실제 KLA 10-K FY2026 손익계산서에서 확인한 값입니다.
(다음 셀에서 문서에 실존하는지 검증합니다.)

In [ ]:
# 4단계: 카테고리별 질문 세트

TEST_QUESTIONS: list[Question] = [
    # ── narrative: 대조군 ──
    Question(id="N1", category="narrative",
             q="What does KLA Corporation do as a business?",
             expect_keywords=["process control", "inspection", "metrology"]),
    Question(id="N2", category="narrative",
             q="What are the main risk factors KLA faces?",
             expect_keywords=["Risk Factors"]),
    Question(id="N3", category="narrative",
             q="Where is KLA headquartered?",
             expect_keywords=["Milpitas"]),

    # ── table_numeric: 표 안의 숫자 ──
    Question(id="T1", category="table_numeric",
             q="What was KLA's total revenue in fiscal year 2026?",
             expect_keywords=["13,579,476"]),
    Question(id="T2", category="table_numeric",
             q="What was KLA's net income in fiscal year 2026?",
             expect_keywords=["4,830,771"]),
    Question(id="T3", category="table_numeric",
             q="What was KLA's research and development expense in fiscal 2026?",
             expect_keywords=["1,532,118"]),
    Question(id="T4", category="table_numeric",
             q="What was KLA's diluted earnings per share in fiscal 2026?",
             expect_keywords=["3.66"]),

    # ── exact_term: 고유명사 / 식별자 ──
    Question(id="E1", category="exact_term",
             q="What is KLA's ticker symbol and which exchange is it listed on?",
             expect_keywords=["KLAC", "Nasdaq"]),
    Question(id="E2", category="exact_term",
             q="What is KLA's IRS Employer Identification Number?",
             expect_keywords=["04-2564110"]),
    Question(id="E3", category="exact_term",
             q="What is KLA's SEC commission file number?",
             expect_keywords=["000-09992"]),
    Question(id="E4", category="exact_term",
             q="What are KLA's reportable business segments?",
             expect_keywords=["PCB", "Semiconductor Process Control"]),

    # ── cross_section: 여러 섹션 결합 ──
    Question(id="C1", category="cross_section",
             q="How do export restrictions affect KLA's business and revenue?",
             expect_keywords=["export", "China"]),
    Question(id="C2", category="cross_section",
             q="What is KLA's capital allocation policy regarding dividends and buybacks?",
             expect_keywords=["dividend", "repurchase"]),

    # ── multi_year: 연도 비교 ──
    Question(id="M1", category="multi_year",
             q="Compare KLA's total revenue between fiscal 2024, 2025, and 2026.",
             expect_keywords=["13,579,476", "12,156,162", "9,812,247"]),
    Question(id="M2", category="multi_year",
             q="How did KLA's service revenue change from fiscal 2024 to 2026?",
             expect_keywords=["3,125,939", "2,329,568"]),
]

import collections

print(f"총 {len(TEST_QUESTIONS)}개\n")
for cat, cnt in collections.Counter(q["category"] for q in TEST_QUESTIONS).items():
    print(f"  {cat:15s} {cnt}개")

### 반드시 먼저 검증해야 할 것

`expect_keywords`에 **문서에 없는 문자열**이 하나라도 있으면,
그 질문은 검색이 아무리 잘 돼도 영원히 실패로 나옵니다.

그러면 "표 검색이 안 되네 → 파서를 바꾸자"는 **잘못된 처방**으로 이어집니다.

평가하기 전에 반드시 확인합니다.

In [ ]:
# 4단계: 정답 키워드가 문서에 실존하는지 검증

missing = []
for q in TEST_QUESTIONS:
    for kw in q["expect_keywords"]:
        if kw not in all_text:
            missing.append((q["id"], kw))

if missing:
    print("[경고] 문서에 없는 키워드 — 질문을 수정하세요:")
    for qid, kw in missing:
        print(f"   {qid}: {kw}")
else:
    print("모든 키워드가 문서에 존재합니다. 평가 진행 가능.")

### 하나 더: 키워드가 너무 흔하면 안 됩니다

예를 들어 `"risks"` 같은 단어는 10-K 곳곳에 나옵니다.
전체 청크의 13%에 들어있다면, **아무 청크나 걸려도 "성공"으로 판정**됩니다.

그러면 점수가 실제보다 부풀려집니다. 변별력을 확인합니다.

In [ ]:
# 4단계: 키워드 변별력 확인 (너무 흔한 키워드는 점수를 부풀린다)

n_chunks = len(chunks)
seen = set()

for q in TEST_QUESTIONS:
    for kw in q["expect_keywords"]:
        if kw in seen:
            continue
        seen.add(kw)
        cnt = sum(1 for c in chunks if kw in c.page_content)
        ratio = cnt / n_chunks
        flag = "  <-- 너무 흔함" if ratio > 0.15 else ""
        print(f"{kw:32s} {cnt:4d}/{n_chunks} ({ratio:5.1%}){flag}")

---

# 5단계: 전체를 돌려서 진단한다

이제 준비가 끝났습니다. 15개를 한 번에 돌립니다.

In [ ]:
# 5단계: 전체 실행

K = 5   # storereg.ipynb의 search_kla_10k와 동일한 k

rows = []
for q in TEST_QUESTIONS:
    r = score(q["q"], q["expect_keywords"], k=K)
    rows.append(dict(id=q["id"], category=q["category"], q=q["q"], **r))

print(f"{'ID':4s} {'카테고리':14s} {'Hit':5s} {'순위':5s} {'RR':6s}")
print("-" * 45)
for row in rows:
    print(f"{row['id']:4s} {row['category']:14s} "
          f"{'O' if row['hit'] else 'X':5s} "
          f"{str(row['rank']) if row['rank'] else '-':5s} "
          f"{row['rr']:<6.3f}")

### 카테고리별로 묶어서 봅니다

전체 평균만 보면 **어느 카테고리가 문제인지 알 수 없습니다.**
처방은 카테고리별로 갈리므로, 반드시 쪼개서 봐야 합니다.

In [ ]:
# 5단계: 카테고리별 집계 — 처방은 여기서 갈린다

n = len(rows)
print(f"전체 Hit Rate@{K} : {sum(r['hit'] for r in rows) / n:.1%}")
print(f"전체 MRR@{K}      : {sum(r['rr'] for r in rows) / n:.3f}")
print()

print(f"{'카테고리':16s} {'Hit Rate':>9s} {'MRR':>7s}   실패")
print("-" * 52)

CATEGORIES = ["narrative", "table_numeric", "exact_term", "cross_section", "multi_year"]

for cat in CATEGORIES:
    rs = [r for r in rows if r["category"] == cat]
    hr = sum(r["hit"] for r in rs) / len(rs)
    mrr_c = sum(r["rr"] for r in rs) / len(rs)
    fails = ",".join(r["id"] for r in rs if not r["hit"]) or "-"
    print(f"{cat:16s} {hr:>8.0%} {mrr_c:>7.3f}   {fails}")

### 이 표를 읽는 법

| 관찰 | 결론 |
|---|---|
| 전부 높음 | 베이스라인 충분 — **처방 불필요** |
| `table_numeric`만 낮음 | 표 파싱 문제 → 레이아웃 파서 / 표 별도 청킹 |
| `exact_term`만 낮음 | 키워드 매칭 부재 → BM25 / 하이브리드 |
| `cross_section`만 낮음 | 청크가 섹션을 자름 → 섹션 / Parent-Child 청킹 |
| Hit은 되는데 순위가 3~5 | 순위 문제 → 재랭킹 |

**한 카테고리가 낮다고 전부 처방하지 마세요.**
과제가 강조하는 건 *"적은 노력으로 문제를 해결"* 입니다.

### 숫자는 "실패했다"만 알려줍니다

**왜** 실패했는지는 검색 결과를 직접 봐야 압니다.
대신 무엇을 가져왔는지 봐야 처방을 고를 수 있습니다.

- 엉뚱한 섹션을 가져왔다 → 청킹 문제
- 맞는 표인데 숫자가 뭉개져 있다 → 파서 문제

In [ ]:
# 5단계: 실패한 질문이 실제로 뭘 가져왔는지 눈으로 본다

def inspect(qid: str, k: int = 5, preview: int = 250):
    q = next(x for x in TEST_QUESTIONS if x["id"] == qid)
    found = vectorstore.similarity_search(q["q"], k=k)

    print("=" * 75)
    print(f"[{q['id']}] {q['q']}")
    print(f"기대 키워드: {q['expect_keywords']}")
    print("=" * 75)

    for i, d in enumerate(found, 1):
        matched = [kw for kw in q["expect_keywords"] if kw in d.page_content]
        flag = f"   <<< 발견: {matched}" if matched else ""
        print(f"\n--- {i}등 (p.{d.metadata.get('page', '?')}){flag}")
        print(d.page_content[:preview].replace("\n", " "))
    print()


failed = [r["id"] for r in rows if not r["hit"]]
print("실패 질문:", failed if failed else "없음")
print()

for qid in failed[:2]:
    inspect(qid)

In [ ]:
# 5단계: 찾긴 했는데 순위가 밀린 질문 = 재랭킹 후보

late = [r for r in rows if r["hit"] and r["rank"] and r["rank"] >= 3]

if late:
    print("정답이 3등 이하로 밀린 질문:")
    for r in late:
        print(f"  {r['id']} ({r['category']}) : {r['rank']}등")
    print("\n-> 재랭킹을 검토할 근거가 됩니다.")
else:
    print("모든 정답이 1~2등에 있습니다.")
    print("-> 재랭킹은 비용 대비 효과가 작습니다. 넣지 마세요.")

---

# 6단계: 검색은 됐는데 답이 틀리는 경우

지금까지는 **검색(retrieval)** 만 봤습니다.
그런데 검색이 성공해도 LLM이 잘못 읽을 수 있습니다.

특히 표에서 흔합니다. FY2026을 물었는데 **옆 열의 FY2025 숫자**를 가져오는 식입니다.
청크 안에 세 연도가 나란히 있으면 충분히 일어납니다.

그래서 agent를 실제로 돌려서 최종 답변도 채점합니다.

In [ ]:
# 6단계: agent 준비 — storereg.ipynb와 동일한 구조

from langchain_core.tools import tool
from langchain.agents import create_agent


@tool
def search_kla_10k(query: str) -> str:
    """Retrieve relevant passages from KLA Corporation's 10-K annual report."""
    found = vectorstore.similarity_search(query, k=5)
    return "\n\n---\n\n".join(d.page_content for d in found)


agent = create_agent(
    model="openai:gpt-5.4-mini",
    tools=[search_kla_10k],
    system_prompt=(
        "You are a financial analyst answering questions about KLA Corporation "
        "using its 10-K filing. Always use the search_kla_10k tool before answering. "
        "Base every factual claim on retrieved content. If the retrieved passages do "
        "not contain the answer, say so explicitly rather than guessing. Answer concisely."
    ),
)

print("agent 준비 완료")

### 답변만 보지 말고, 도구 호출도 같이 기록합니다

과제 5번에서 이렇게 경고합니다.

> *"retriever는 완벽한데 agent가 도구를 아예 안 부르거나 엉뚱한 쿼리로 부르는"* 문제

그래서 `ask_agent`는 두 가지를 같이 기록합니다.
- 최종 답변 (정답 포함 여부)
- agent가 **어떤 쿼리로** 도구를 불렀는지

In [ ]:
# 6단계: agent 실행 + 도구 호출 쿼리 기록

def ask_agent(question: Question):
    state = agent.invoke({"messages": [{"role": "user", "content": question["q"]}]})

    tool_queries = []
    for m in state["messages"]:
        for call in getattr(m, "tool_calls", None) or []:
            tool_queries.append(call["args"].get("query", ""))

    answer = state["messages"][-1].content
    if isinstance(answer, list):   # 일부 모델은 content가 블록 리스트
        answer = " ".join(b.get("text", "") for b in answer if isinstance(b, dict))

    correct = any(kw in answer for kw in question["expect_keywords"])
    return dict(answer=answer, tool_queries=tool_queries, correct=correct)


# 먼저 1개만 테스트 (전체 돌리기 전에 동작 확인)
_t = ask_agent(TEST_QUESTIONS[3])
print("정답 여부 :", _t["correct"])
print("도구 쿼리 :", _t["tool_queries"])
print("답변      :", _t["answer"][:200])

### 전체 15개를 돌립니다

**LLM을 15번 호출하므로 3~6분 걸립니다.**
시간이 없으면 이 셀은 건너뛰어도 5단계까지의 진단은 유효합니다.

In [ ]:
# 6단계: 전체 실행 (3~6분 소요)

gen_rows = []
for q in TEST_QUESTIONS:
    res = ask_agent(q)
    gen_rows.append(dict(id=q["id"], category=q["category"], **res))
    print(f"[{'O' if res['correct'] else 'X'}] {q['id']:3s} | "
          f"도구 {len(res['tool_queries'])}회 | "
          f"{res['answer'][:60].replace(chr(10), ' ')}")

### 검색과 생성을 교차해서 봅니다

이 표가 **이 노트북에서 가장 중요한 출력**입니다.

| 검색 | 답변 | 의미 |
|---|---|---|
| O | O | 정상 |
| O | X | 근거는 찾았는데 잘못 읽음 → **생성 문제** |
| **X** | **O** | **근거 없이 맞춤 → 환각 의심** |
| X | X | 검색 실패 → 처방 대상 |

### `검색 X / 답변 O`가 왜 위험한가

LLM이 **사전학습 지식으로** 답을 맞춘 것입니다. RAG가 작동한 게 아닙니다.

KLA처럼 유명한 회사에서는 자주 일어납니다.
그리고 **덜 알려진 회사로 문서를 바꾸는 순간 전부 틀립니다.**

지금 "값이 잘 나온다"고 느끼시는 것 중 일부가 여기 해당할 수 있습니다.
**이 건수를 리포트에 반드시 적으세요.**

In [ ]:
# 6단계: 검색 vs 생성 교차 분석

ret_by_id = {r["id"]: r for r in rows}

print(f"{'ID':4s} {'카테고리':14s} {'검색':5s} {'답변':5s}  진단")
print("-" * 70)

counter = collections.Counter()
for g in gen_rows:
    r_ok = bool(ret_by_id[g["id"]]["hit"])
    g_ok = g["correct"]

    if r_ok and g_ok:
        diag = "정상"
    elif r_ok and not g_ok:
        diag = "생성 문제 (근거는 찾았는데 잘못 읽음)"
    elif not r_ok and g_ok:
        diag = "환각 의심 (근거 없이 맞춤)"
    else:
        diag = "검색 실패 -> 처방 대상"

    counter[diag] += 1
    print(f"{g['id']:4s} {g['category']:14s} "
          f"{'O' if r_ok else 'X':5s} {'O' if g_ok else 'X':5s}  {diag}")

print("\n요약")
for d, c in counter.most_common():
    print(f"  {c:2d}건  {d}")

In [ ]:
# 6단계: agent가 자연어 질문을 어떤 검색 쿼리로 바꿨는지 확인

for g in gen_rows:
    q = next(x for x in TEST_QUESTIONS if x["id"] == g["id"])
    print(f"[{g['id']}] {q['q']}")
    if not g["tool_queries"]:
        print("     !! 도구 호출 안 함 -> 프롬프트 문제")
    for tq in g["tool_queries"]:
        print(f"     -> {tq}")
    print()

---

# 7단계: 한국어로 물어보면?

과제 6번 항목입니다. 문서는 영어인데 질문이 한국어로 들어오는 경우입니다.

## 검증할 가설

> `text-embedding-3-small`은 멀티링구얼이므로,
> 한국어 질문으로도 영어 청크를 찾아낼 것이다.

**이걸 먼저 확인해야 하는 이유:**

가설이 참이면 → 번역 레이어를 만들 필요가 없습니다.
거짓이면 → 번역이나 가중치 조정이 필요합니다.

과제가 제시한 선택지 3번 *"dense만으로 충분한지 먼저 검증"* 이 이것입니다.
**측정도 안 하고 번역기부터 붙이는 건 과잉 설계입니다.**

같은 질문을 한국어/영어로 각각 던져서 비교합니다.

In [ ]:
# 7단계: 같은 질문을 한/영으로 던져서 비교

KO_EN_PAIRS = [
    ("KLA의 2026 회계연도 총매출은 얼마인가요?",
     "What was KLA's total revenue in fiscal year 2026?", ["13,579,476"]),
    ("KLA의 2026 회계연도 순이익은 얼마인가요?",
     "What was KLA's net income in fiscal year 2026?", ["4,830,771"]),
    ("KLA 본사는 어디에 있나요?",
     "Where is KLA headquartered?", ["Milpitas"]),
    ("KLA가 직면한 주요 리스크는 무엇인가요?",
     "What are the main risk factors KLA faces?", ["Risk Factors"]),
    ("KLA의 주식 티커 심볼은 무엇인가요?",
     "What is KLA's ticker symbol?", ["KLAC"]),
    ("KLA의 사업 부문은 어떻게 나뉘나요?",
     "What are KLA's reportable business segments?",
     ["PCB", "Semiconductor Process Control"]),
]

print(f"{'한국어':14s} {'영어':14s}  판정")
print("-" * 46)

ko_rr, en_rr = [], []
for ko, en, kws in KO_EN_PAIRS:
    rk = score(ko, kws)
    re_ = score(en, kws)
    ko_rr.append(rk["rr"])
    en_rr.append(re_["rr"])

    ks = f"{'HIT ' + str(rk['rank']) + '등' if rk['hit'] else 'MISS'}"
    es = f"{'HIT ' + str(re_['rank']) + '등' if re_['hit'] else 'MISS'}"
    verdict = "동일" if rk["rr"] == re_["rr"] else ("한국어 열세" if rk["rr"] < re_["rr"] else "한국어 우세")
    print(f"{ks:14s} {es:14s}  {verdict}")

m = len(KO_EN_PAIRS)
print(f"\n한국어 MRR : {sum(ko_rr)/m:.3f}")
print(f"영어   MRR : {sum(en_rr)/m:.3f}")

In [ ]:
# 7단계: 격차를 보고 판단한다

gap = (sum(en_rr) - sum(ko_rr)) / m
print(f"MRR 격차 (영어 - 한국어): {gap:+.3f}\n")

if gap < 0.05:
    print("교차언어 검색이 사실상 동등하게 작동합니다.")
    print(" -> dense 단독으로 한국어 질문 처리 가능. 번역 레이어 불필요.")
    print(" -> BM25를 추가하더라도 한국어 질문에서는 sparse 가중치를 낮추는 게 합리적입니다.")
elif gap < 0.2:
    print("한국어가 약간 열세이나 실용 범위입니다.")
    print(" -> 우선 dense 단독 유지. 어떤 질문 유형이 실패했는지 먼저 확인하세요.")
else:
    print("한국어 검색 성능이 뚜렷하게 떨어집니다.")
    print(" -> 질문을 영어로 번역 후 검색하는 레이어를 검토하세요.")
    print(" -> 단, 고유명사/숫자 왜곡 위험이 있으니 번역 전후를 다시 측정할 것.")

### 리포트에 쓸 내용

과제는 *"두 가지 방식을 다 테스트해보고 직접 비교"* 하라고 했습니다.

여기서 dense만으로 충분하다는 결과가 나오면,
번역 레이어를 안 만든 것이 **게으름이 아니라 측정에 근거한 판단**이 됩니다.

"안 만들었습니다"보다 **"측정해보니 격차가 0.02라 불필요하다고 판단했습니다"** 가
훨씬 좋은 평가를 받습니다.

---

# 8단계: 진단 결과를 처방으로 연결한다

지금까지 측정한 숫자를 과제의 처방 메뉴에 매칭합니다.

**임계치는 `0.7`로 잡았습니다.**
절대적인 기준은 아닙니다. 질문 난이도에 따라 조정하세요.

In [ ]:
# 8단계: 측정값 -> 처방 자동 매칭

THRESHOLD = 0.7


def cat_hit(cat: str):
    rs = [r for r in rows if r["category"] == cat]
    return sum(r["hit"] for r in rs) / len(rs) if rs else None


prescriptions = []

checks = [
    ("table_numeric", "표 안의 숫자를 못 찾음",
     "레이아웃 인식 파서(LlamaParse/Docling)로 교체 + 표를 별도 청크로 분리"),
    ("exact_term", "고유명사/식별자를 못 찾음",
     "BM25 추가 후 EnsembleRetriever로 하이브리드 검색"),
    ("cross_section", "섹션이 잘려 문맥이 끊김",
     "Item 단위 섹션 청킹 또는 Parent-Child 청킹"),
    ("multi_year", "연도 비교 실패",
     "표 처방 우선 적용 (연도별 열이 한 청크에 유지되어야 함)"),
]

for cat, symptom, rx in checks:
    hr = cat_hit(cat)
    if hr is not None and hr < THRESHOLD:
        prescriptions.append((symptom, f"{cat} Hit Rate {hr:.0%}", rx))

# 재랭킹 판단
hits = [r for r in rows if r["hit"] and r["rank"]]
if hits:
    late_ratio = sum(1 for r in hits if r["rank"] >= 3) / len(hits)
    if late_ratio >= 0.3:
        prescriptions.append((
            "정답이 상위권에 못 옴",
            f"Hit 중 {late_ratio:.0%}가 3등 이하",
            "Cross-Encoder Reranker 추가 (k를 크게 뽑고 재정렬)",
        ))

print("=" * 70)
print("진단 결과")
print("=" * 70)

if not prescriptions:
    print("\n임계치를 넘는 증상이 없습니다.")
    print("이 질문 세트에서는 베이스라인이 충분합니다.")
    print("-> 처방을 적용하지 않는 것도 정당한 결론입니다.")
    print("-> 다만 질문 난이도를 올려 다시 측정해보길 권합니다.")
else:
    for i, (sym, ev, rx) in enumerate(prescriptions, 1):
        print(f"\n{i}. 증상 : {sym}")
        print(f"   근거 : {ev}")
        print(f"   처방 : {rx}")

print("\n" + "=" * 70)

### 처방이 하나도 안 나왔다면

그것도 **유효한 결과**입니다. 과제는 모든 기법을 쓰라고 하지 않았습니다.

다만 질문 세트가 쉬웠을 가능성이 큽니다. 난이도를 올려보세요.

- 표의 **각주**에 있는 숫자 (특정 세그먼트의 감가상각비 등)
- 여러 표를 **계산**해야 나오는 값 (영업이익률 등)
- 10-K 뒷부분 **Exhibit 목록**의 특정 문서 날짜
- **부정형** 질문 ("KLA가 보고하지 **않는** 세그먼트는?")

---

# 9단계: 리포트용 요약

아래 출력을 그대로 리포트에 옮길 수 있습니다.

In [ ]:
# 9단계: 리포트용 요약 출력

hit_rate = sum(r["hit"] for r in rows) / len(rows)
mrr = sum(r["rr"] for r in rows) / len(rows)

print("#" * 68)
print("# 10-K RAG 베이스라인 평가 결과")
print("#" * 68)

print(f"""
## 1. 베이스라인 구성
- 파서       : PyPDFLoader (단순 텍스트 추출)
- 청킹       : RecursiveCharacterTextSplitter (size=1000, overlap=200)
- 임베딩     : text-embedding-3-small
- 벡터스토어 : InMemoryVectorStore (코사인 유사도)
- 검색       : similarity_search(k={K})
- 문서       : KLA 10-K FY2026, {len(docs)}페이지 -> {len(chunks)}청크

## 2. 평가 방법
- 질문 {len(TEST_QUESTIONS)}개를 5개 카테고리로 설계 (narrative는 대조군)
- Retrieval : Hit Rate@{K}, MRR@{K}
  (정답 근거 청크에 반드시 포함될 키워드 매칭으로 자동 판정)
- Generation: 최종 답변 내 정답 문자열 포함 여부
- 교차언어  : 동일 질문 한/영 쌍 {len(KO_EN_PAIRS)}개 비교

## 3. 측정 결과
- 전체 Hit Rate@{K} : {hit_rate:.1%}
- 전체 MRR@{K}      : {mrr:.3f}""")

print("\n- 카테고리별:")
for cat in CATEGORIES:
    rs = [r for r in rows if r["category"] == cat]
    hr = sum(r["hit"] for r in rs) / len(rs)
    mr = sum(r["rr"] for r in rs) / len(rs)
    print(f"    {cat:16s} Hit {hr:>4.0%} | MRR {mr:.3f}")

_ok = sum(1 for g in gen_rows if ret_by_id[g["id"]]["hit"] and g["correct"])
_gen_bad = sum(1 for g in gen_rows if ret_by_id[g["id"]]["hit"] and not g["correct"])
_halluc = sum(1 for g in gen_rows if not ret_by_id[g["id"]]["hit"] and g["correct"])
_both = sum(1 for g in gen_rows if not ret_by_id[g["id"]]["hit"] and not g["correct"])

print(f"""
- 검색/생성 교차분석:
    검색 O + 답변 O : {_ok:2d}건  (정상)
    검색 O + 답변 X : {_gen_bad:2d}건  (생성 단계 문제)
    검색 X + 답변 O : {_halluc:2d}건  (환각 의심)
    검색 X + 답변 X : {_both:2d}건  (검색 실패)

- 교차언어:
    한국어 MRR {sum(ko_rr)/m:.3f} vs 영어 MRR {sum(en_rr)/m:.3f} (격차 {gap:+.3f})

## 4. 진단 -> 적용할 처방""")

if not prescriptions:
    print("    임계치를 넘는 증상 없음. 처방 미적용이 타당함.")
else:
    for i, (sym, ev, rx) in enumerate(prescriptions, 1):
        print(f"    {i}) {sym}")
        print(f"       근거: {ev}")
        print(f"       처방: {rx}")

print("\n" + "#" * 68)

---

# 다음 단계

이 노트북은 **진단까지만** 합니다. 처방은 별도 노트북에서 적용하세요.

## 처방 적용 시 규칙

**1. 한 번에 하나씩만 적용합니다.**
BM25와 재랭킹을 동시에 넣으면 어느 쪽이 효과를 냈는지 알 수 없습니다.

**2. 적용 후 5단계를 그대로 다시 돌립니다.**
같은 `TEST_QUESTIONS`, 같은 `score()`로 측정해야 비교가 성립합니다.
그래서 이 노트북이 질문 세트와 채점 함수를 **고정**해둔 것입니다.

**3. 전/후를 나란히 기록합니다.**

| 구성 | Hit Rate@5 | MRR@5 | table_numeric | exact_term |
|---|---|---|---|---|
| 베이스라인 | ? | ? | ? | ? |
| + 처방 1 | ? | ? | ? | ? |
| + 처방 2 | ? | ? | ? | ? |

**4. 효과가 없으면 되돌립니다.**
숫자가 안 오르는 처방은 복잡도만 늘립니다. 이것도 리포트에 쓸 내용입니다.

## 과제 5번 (LangGraph 배포)로 넘어갈 때

확정된 파이프라인을 `src/tools/rag_tool.py`로 옮기고 `langgraph dev`로 띄운 뒤,
자연어 질문으로 agent 통합 테스트를 합니다.

6단계에서 기록한 **"질문 -> 도구 쿼리 변환"** 출력이
그 통합 테스트의 사전 점검 역할을 합니다.